# Vector stores and semantic search



In [61]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np

## Part I: Basic vector store implementation

In [62]:
import numpy as np

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        new_embs = self.model.encode([doc.text for doc in documents], convert_to_numpy=True)
        self.embeddings.extend(new_embs)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []
        query_emb = self.model.encode([query], convert_to_numpy=True)
        scores = util.cos_sim(query_emb, self.embeddings)[0].numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [SearchResult(float(scores[i]), self.documents[i]) for i in top_idx]

In [63]:
df = pd.read_csv("Animal_Fun_Facts/animal-fun-facts-dataset.csv").fillna("")

documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name": row["animal_name"],
            "source": row["source"],
            "media_link": row["media_link"],
            "wikipedia_link": row["wikipedia_link"],
        }
    )
    for _, row in df.iterrows()
    if row["text"].strip()       
]

In [64]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)
store.add_documents(documents)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8646.15it/s]


In [65]:
queries = [
    "are the only mammals with wings",
    "mammals that can fly",
    "16 hours a day sleeping",
    "tongue is about 18 inches long",
    "reptiles and cold-blooded creatures",
]

In [66]:
def run_queries(queries: list[str], top_k: int = 5):
    for q in queries:
        print(f"\n{'='*60}\nQuery: {q}\n{'='*60}")

        for i, r in enumerate(store.search(q, top_k=top_k), 1):
            print(f"\n[{i}] Score: {r.score:.4f}")
            print(f"Text: {r.document.text[:120].strip()}...")
            for k, v in r.document.metadata.items():
                if v:
                    print(f"{k}: {v}")

run_queries(queries)


Query: are the only mammals with wings

[1] Score: 0.8304
Text: Bats are the only mammals with wings, and the only ones that can truly fly...
animal_name: bat
source: https://www.animalfactsencyclopedia.com/Bat-facts.html
wikipedia_link: /wiki/Bat

[2] Score: 0.7139
Text: Bats are the world's only flying mammals. Other mammals may glide through the air, but bats flap their wings and fly....
animal_name: malayan flying fox
source: https://seaworld.org/animals/facts/mammals/malayan-flying-fox/
wikipedia_link: /wiki/Large_flying_fox

[3] Score: 0.6941
Text: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works i...
animal_name: colugo (flying lemur)
source: https://factanimal.com/colugo/
wikipedia_link: /wiki/Colugo

[4] Score: 0.6743
Text: Bats are the only flying mammals and comprise the second largest order of mammals in the world....
animal_name: bats
source: https://seaworld.org/animals/facts/mammals/bats/
wikipedia_li

## Part II: Filtering by metadata

In [67]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []


    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        new_embs = self.embedding_model.encode([doc.text for doc in documents], convert_to_numpy=True)
        self.embeddings.extend(new_embs)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if metadata_filter:
            candidates = [i for i, doc in enumerate(self.documents)
                          if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())]
            filtered_embs = [self.embeddings[i] for i in candidates]
        else:
            candidates = list(range(len(self.documents)))
            filtered_embs = self.embeddings

        query_emb = self.embedding_model.encode([query], convert_to_numpy=True)
        scores = util.cos_sim(query_emb, filtered_embs)[0].numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [SearchResult(float(scores[j]), self.documents[candidates[j]]) for j in top_idx]

In [68]:
df = pd.read_csv("SpotifyReviews/reviews.csv").fillna("")

documents = [
    Document(
        text=row["Review"],
        metadata={
            "rating": str(row["Rating"]),
            "time": str(row["Time_submitted"]),
            "thumbsup": str(row["Total_thumbsup"]),
        }
    )
    for _, row in df.iterrows()
    if row["Review"].strip()
]

In [69]:
fstore = FilteredVectorStore(model)
fstore.add_documents(documents)

In [70]:
queries_with_filters = [
    ("app crashes and freezes", None),
    ("great music experience", {"rating": "5"}),
    ("ads are annoying", {"rating": "1"}),
    ("battery drain and slow performance", {"rating": "1"}),
    ("best music app I have ever used", {"rating": "5"}),
    ("decent music app but has some issues", {"rating": "3"}),
    ("I love this app", {"thumbsup": "5"}),
]

In [71]:
def run_filtered_queries(queries: list[tuple[str, dict | None]], top_k: int = 5):
    for query, metadata_filter in queries:
        label = f"filter: {metadata_filter}" if metadata_filter else "sin filtro"
        print(f"\n{'='*60}\nQuery: {query} | {label}\n{'='*60}")
        for i, r in enumerate(fstore.search(query, top_k=top_k, metadata_filter=metadata_filter), 1):
            print(f"[{i}] Score: {r.score:.4f} | {r.document.text[:100]}...")
            for k, v in r.document.metadata.items():
                if v:
                    print(f"{k}: {v}")

run_filtered_queries(queries_with_filters)


Query: app crashes and freezes | sin filtro
[1] Score: 0.9695 | app constantly crashes and freezes...
rating: 1
time: 2022-05-29 19:47:34
thumbsup: 0
[2] Score: 0.9441 | App is crashed and freezed...
rating: 1
time: 2022-06-15 14:07:12
thumbsup: 0
[3] Score: 0.9038 | App crashes all the time...
rating: 1
time: 2022-04-21 00:11:54
thumbsup: 0
[4] Score: 0.8956 | App crashes constantly...
rating: 1
time: 2022-07-07 14:46:27
thumbsup: 0
[5] Score: 0.8880 | app always crash...
rating: 1
time: 2022-05-30 14:49:45
thumbsup: 0

Query: great music experience | filter: {'rating': '5'}
[1] Score: 0.9643 | Great music experience....
rating: 5
time: 2022-06-18 14:12:52
thumbsup: 0
[2] Score: 0.9639 | Excellent music experience...
rating: 5
time: 2022-07-09 07:52:04
thumbsup: 0
[3] Score: 0.9407 | A wonderful music experience...
rating: 5
time: 2022-06-28 08:36:48
thumbsup: 0
[4] Score: 0.9168 | Brilliant music experience....
rating: 5
time: 2022-07-07 17:32:54
thumbsup: 0
[5] Score: 0.9145 | Grea

## Conclusiones

Es una buena actividad para aplicar lo que se vio en clase sobre Vector Stores y búsqueda semántica. Tener la base fue suficiente para desarrollar el notebook capaz de realizar consultas sobre conjuntos de datos utilizando la búsqueda semántica para encontrar el contenido por sus significado y no por palabras exactas que un documento debe tener.

Además con la versión de filtro se pueden recuperar los documentos de manera más precisa para lo que se necesite, obteniendo solo lo que importa para los parámetros que se dan. Todo en conjunto es un concepto muy interesante que puede tener muchas aplicaciones en diferentes áreas.